[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/transformer_self_attention.ipynb)

# Self-attention and the Transformer block, step by step

This notebook follows the *Transformers* slides. Every formula is computed with small numbers you can check by hand, and then compared with what PyTorch and a real pre-trained model do:

1. **Self-attention by hand**: the *the blue house* worked example from the slides.
2. **Matrix form** and shapes.
3. **Why divide by $\sqrt{d_k}$?**
4. **Causal masking**.
5. **Multi-head attention**, checked against `nn.MultiheadAttention`.
6. **Positional encoding**: self-attention ignores order.
7. **The Transformer block**: residuals, layer normalisation, feed-forward network.
8. **The cost**: quadratic in the sequence length.
9. **Exercises** from the slides.
10. **Back to *it***: self-attention in a pre-trained BERT.

The companion notebook `transformer_translation` builds a complete Transformer with these pieces and trains it.

### Notation (the same as in the slides)

Rows are tokens: a sequence of $n$ tokens is a matrix with $n$ rows.

| symbol | meaning | shape |
|---|---|---|
| $x_i$ | input vector of token $i$ (embedding + position) | $d_\text{model}$ |
| $X$ | all the input vectors, one per row | $n \times d_\text{model}$ |
| $W^Q, W^K, W^V$ | learned projection matrices | $d_\text{model} \times d_k$ (or $d_v$) |
| $q_i, k_i, v_i$ | query, key and value of token $i$; rows of $Q, K, V$ | $d_k$, $d_k$, $d_v$ |
| $\alpha_{ij}$ | attention weight that token $i$ gives to token $j$; rows of $A$ | $A$: $n \times n$ |
| $z_i$ | output of attention for token $i$; rows of $Z$ | $Z$: $n \times d_v$ |
| $h$, $N$ | number of heads, number of stacked blocks | |

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

np.set_printoptions(precision=2, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)


def heatmap(M, x_labels, y_labels, title="", ax=None, vmin=0, vmax=1, fmt="{:.2f}"):
    # Matrix as a heatmap with its values written in the cells.
    ax = ax or plt.figure(figsize=(0.75 * len(x_labels) + 1.5, 0.55 * len(y_labels) + 1)).gca()
    ax.imshow(M, cmap="Blues", vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]):
                ax.text(j, i, fmt.format(M[i, j]), ha="center", va="center", fontsize=8,
                        color="white" if M[i, j] > vmin + 0.6 * (vmax - vmin) else "black")
    ax.set_xticks(range(len(x_labels)), x_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(y_labels)), y_labels)
    ax.set_title(title)
    return ax

## 1. Self-attention by hand: *the blue house*

In classic attention (previous lecture), the **decoder** looked back at the **encoder**: the query came from one sequence and the keys and values from another. In **self-attention**, every token of a sequence attends to every token of the **same** sequence, including itself. Each token plays three roles, through three learned projections:

$$q_i = x_i W^Q, \qquad k_i = x_i W^K, \qquad v_i = x_i W^V$$

$$\text{score}(i, j) = \frac{q_i \cdot k_j}{\sqrt{d_k}}, \qquad \alpha_{ij} = \frac{\exp(\text{score}(i,j))}{\sum_{l=1}^{n} \exp(\text{score}(i,l))}, \qquad z_i = \sum_{j=1}^{n} \alpha_{ij}\, v_j$$

The worked example of the slides: $n = 3$ tokens, $d_\text{model} = d_k = d_v = 2$, and, to keep the arithmetic short, $W^K = W^V = I$ (so $K = V = X$).

In [ ]:
tokens = ["the", "blue", "house"]
X = np.array([[1.0, 0.0],     # the
              [0.0, 1.0],     # blue
              [1.0, 1.0]])    # house

W_Q = np.array([[0.0, 1.0],
                [1.0, 1.0]])
W_K = np.eye(2)
W_V = np.eye(2)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V
print("Q = X W^Q =\n", Q)
print("q_house = (1, 1) W^Q =", X[2] @ W_Q)

**Step 1: scores.** The dot product of every query with every key: entry $(i, j)$ of $QK^\top$ is $q_i \cdot k_j$. Then divide by $\sqrt{d_k} = \sqrt{2}$.

In [ ]:
d_k = Q.shape[1]
scores = Q @ K.T
scaled = scores / np.sqrt(d_k)
print("Q K^T =\n", scores)
print("Q K^T / sqrt(2) =\n", scaled)

**Step 2: softmax of each row.** Row $i$ says how token $i$ distributes its attention over the $n$ tokens, so each **row** sums to 1 (the columns do not have to).

In [ ]:
def softmax_rows(S):
    S = S - S.max(axis=1, keepdims=True)     # does not change the result, avoids overflow
    E = np.exp(S)
    return E / E.sum(axis=1, keepdims=True)

A = softmax_rows(scaled)
print("A =\n", A)
print("row sums:", A.sum(axis=1), "  column sums:", A.sum(axis=0))
heatmap(A, tokens, tokens, "A: row i = how token i distributes its attention")
plt.xlabel("key j")
plt.ylabel("query i")
plt.show()

**Step 3: outputs.** Each output is a weighted average of the **values**: $Z = AV$.

In [ ]:
Z = A @ V
print("Z = A V =\n", Z)

# The same for one token, written as the sum of the slides
z_house = sum(A[2, j] * V[j] for j in range(3))
print("z_house = " + " + ".join(f"{A[2, j]:.2f}*{V[j]}" for j in range(3)), "=", z_house)

**Explain:**

- The input vector of *house* was $(1, 1)$; its output is $(0.72, 0.86)$. Where did the change come from?
- Every $z_i$ now contains information from the **whole** sentence. This is a **contextual embedding**: the output of a token depends on the other tokens.
- Here $W^Q$ was chosen by hand and $W^K = W^V = I$. In a real model, the three matrices are **learned**, so that the weights are useful (e.g. *it* → *animal*).

## 2. Matrix form and shapes

All tokens at once, with only matrix products:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V, \qquad \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

From now on we use PyTorch, with a larger random example: $n = 5$ tokens, $d_\text{model} = 8$, $d_k = d_v = 4$.

In [ ]:
def self_attention(X, W_q, W_k, W_v, mask=None):
    # X: (n, d_model).  mask: (n, n) of booleans, True = allowed.  Returns Z (n, d_v) and A (n, n).
    Q, K, V = X @ W_q, X @ W_k, X @ W_v
    scores = Q @ K.T / math.sqrt(K.shape[1])        # (n, n)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    A = F.softmax(scores, dim=1)                      # softmax of each row
    return A @ V, A

torch.manual_seed(0)
n, d_model, d_k = 5, 8, 4
X5 = torch.randn(n, d_model)
W_q, W_k, W_v = (torch.randn(d_model, d_k) for _ in range(3))

Z5, A5 = self_attention(X5, W_q, W_k, W_v)
print("X:", tuple(X5.shape), " W:", tuple(W_q.shape), " Q, K, V:", tuple((X5 @ W_q).shape),
      " QK^T and A:", tuple(A5.shape), " Z:", tuple(Z5.shape))

# The same, token by token, with the formulas of section 1
for i in range(n):
    q_i = X5[i] @ W_q
    s = torch.stack([q_i @ (X5[j] @ W_k) / math.sqrt(d_k) for j in range(n)])
    alpha = torch.exp(s) / torch.exp(s).sum()
    z_i = sum(alpha[j] * (X5[j] @ W_v) for j in range(n))
    assert torch.allclose(z_i, Z5[i], atol=1e-5)
print("matrix form == token-by-token loop")

**Explain:** the loop computes the $n$ outputs one after the other, but no iteration needs the result of another. The matrix form computes them **in parallel**, which is what GPUs are good at. An RNN cannot do this: $a^{\langle t \rangle}$ needs $a^{\langle t-1 \rangle}$.

## 3. Why divide by $\sqrt{d_k}$?

If the components of $q$ and $k$ are independent with mean 0 and variance 1, then $q \cdot k = \sum_{m=1}^{d_k} q_m k_m$ has mean 0 and **variance $d_k$**.

**Predict first:** with $d_k = 64$, what is the standard deviation of the scores? What does the softmax do with scores that large?

In [ ]:
rng = np.random.default_rng(0)
dims = [4, 16, 64, 256, 512]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for d in [4, 64, 512]:
    q = rng.standard_normal((20000, d))
    k = rng.standard_normal((20000, d))
    axes[0].hist((q * k).sum(axis=1), bins=100, histtype="step", density=True, label=f"$d_k$={d}")
    axes[1].hist((q * k).sum(axis=1) / np.sqrt(d), bins=100, histtype="step", density=True, label=f"$d_k$={d}")
axes[0].set_title("q · k")
axes[1].set_title("q · k / √d_k")
axes[0].legend()
plt.show()

print(f"{'d_k':>5s} {'std(q·k)':>9s} {'max α':>7s} {'max α (÷√d_k)':>14s} {'|∂α/∂score|':>12s} {'(÷√d_k)':>8s}")
for d in dims:
    q = rng.standard_normal((2000, d))
    k = rng.standard_normal((2000, 10, d))                     # each query against 10 keys
    s = np.einsum("nd,ntd->nt", q, k)
    rows = []
    for scale in [1, np.sqrt(d)]:
        a = softmax_rows(s / scale)
        # Jacobian of the softmax: diag(a) - a a^T. Its size says how much gradient flows back.
        jac = np.array([np.linalg.norm(np.diag(r) - np.outer(r, r)) for r in a])
        rows += [a.max(axis=1).mean(), jac.mean()]
    print(f"{d:5d} {s.std():9.1f} {rows[0]:7.2f} {rows[2]:14.2f} {rows[1]:12.3f} {rows[3]:8.3f}")

**Explain:** without scaling, the scores grow like $\sqrt{d_k}$; the softmax **saturates** (one weight close to 1, the others close to 0), and its derivative becomes close to 0, so almost no gradient flows back to $W^Q$ and $W^K$. Dividing by $\sqrt{d_k}$ brings the variance back to 1, whatever the size.

## 4. Causal masking

When generating text, token $i$ may only attend to tokens $j \le i$: the following tokens do not exist yet. A mask $M$ is added to the scores before the softmax:

$$A = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right), \qquad M_{ij} = \begin{cases} 0 & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

In code it is easier to keep a boolean matrix (True = allowed) and use `masked_fill`. A lower-triangular matrix is exactly "$j \le i$":

In [ ]:
causal = torch.tril(torch.ones(n, n, dtype=torch.bool))
print(causal.int())

Z_masked, A_masked = self_attention(X5, W_q, W_k, W_v, mask=causal)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
labels = [f"tok {i+1}" for i in range(n)]
heatmap(A5.numpy(), labels, labels, "no mask", ax=axes[0])
heatmap(A_masked.numpy(), labels, labels, "causal mask", ax=axes[1])
plt.tight_layout()
plt.show()

**Predict first:** with the causal mask, if we change the last two tokens of the input, which outputs $z_i$ change?

In [ ]:
X_changed = X5.clone()
X_changed[3:] = torch.randn(2, d_model)          # new tokens 4 and 5

Z_changed, _ = self_attention(X_changed, W_q, W_k, W_v, mask=causal)
for i in range(n):
    print(f"z_{i+1} unchanged? {torch.allclose(Z_masked[i], Z_changed[i])}")

**Explain:** $z_1, z_2, z_3$ only depend on the tokens up to themselves. This is what makes training in parallel possible: in **one** pass over a whole sentence, the output at position $i$ is exactly what the model would compute if the sentence stopped at $i$. So all the "predict the next token" problems of a sentence are solved at once.

## 5. Multi-head attention

One attention layer computes one kind of relation between tokens. **Multi-head attention** runs $h$ heads in parallel, each with its own $W^Q_i, W^K_i, W^V_i$ of size $d_\text{model} \times d_k$, with $d_k = d_\text{model}/h$:

$$\text{head}_i = \text{Attention}(XW^Q_i,\; XW^K_i,\; XW^V_i), \qquad \text{MultiHead}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^O$$

In practice, the $h$ matrices $W^Q_i$ are stored side by side in one $d_\text{model} \times d_\text{model}$ matrix, and the result is **split** into heads with a reshape. Both ways give the same result:

In [ ]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, h, mask=None):
    # X: (n, d_model).  W_q, W_k, W_v, W_o: (d_model, d_model).  Returns Z (n, d_model) and A (h, n, n).
    n, d_model = X.shape
    d_k = d_model // h
    # (n, d_model) -> (n, h, d_k) -> (h, n, d_k): one slice of d_k columns per head
    Q = (X @ W_q).view(n, h, d_k).transpose(0, 1)
    K = (X @ W_k).view(n, h, d_k).transpose(0, 1)
    V = (X @ W_v).view(n, h, d_k).transpose(0, 1)
    scores = Q @ K.transpose(1, 2) / math.sqrt(d_k)            # (h, n, n): one score matrix per head
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    A = F.softmax(scores, dim=-1)
    heads = A @ V                                               # (h, n, d_k)
    concat = heads.transpose(0, 1).reshape(n, d_model)          # Concat(head_1, ..., head_h)
    return concat @ W_o, A

torch.manual_seed(0)
h = 2
W_q8, W_k8, W_v8, W_o8 = (torch.randn(d_model, d_model) / math.sqrt(d_model) for _ in range(4))
Z_mh, A_mh = multi_head_attention(X5, W_q8, W_k8, W_v8, W_o8, h)

# The same with one self_attention call per head, using the columns of that head
d_k = d_model // h
cols = [slice(i * d_k, (i + 1) * d_k) for i in range(h)]
heads = [self_attention(X5, W_q8[:, c], W_k8[:, c], W_v8[:, c])[0] for c in cols]
Z_loop = torch.cat(heads, dim=1) @ W_o8
print("Z:", tuple(Z_mh.shape), "  A:", tuple(A_mh.shape))
print("reshape version == one head at a time:", torch.allclose(Z_mh, Z_loop, atol=1e-5))

PyTorch has this layer ready: `nn.MultiheadAttention`. It stores $W^Q, W^K, W^V$ stacked in one `in_proj_weight` and $W^O$ in `out_proj`, transposed (PyTorch layers compute $xW^\top$). If we copy our matrices into it, it must give exactly the same result:

In [ ]:
mha = nn.MultiheadAttention(d_model, num_heads=h, bias=False, batch_first=True)
with torch.no_grad():
    mha.in_proj_weight.copy_(torch.cat([W_q8.T, W_k8.T, W_v8.T]))
    mha.out_proj.weight.copy_(W_o8.T)

X_batch = X5.unsqueeze(0)                                       # (batch=1, n, d_model)
Z_torch, A_torch = mha(X_batch, X_batch, X_batch, average_attn_weights=False)
print("same outputs as nn.MultiheadAttention:", torch.allclose(Z_mh, Z_torch[0], atol=1e-5))
print("same attention weights:", torch.allclose(A_mh, A_torch[0], atol=1e-5))

fig, axes = plt.subplots(1, h, figsize=(10, 3.5))
for i in range(h):
    heatmap(A_mh[i].detach().numpy(), labels, labels, f"head {i+1}", ax=axes[i])
plt.tight_layout()
plt.show()

**Explain:** the two heads see the same tokens but produce different attention patterns, because their projections are different. After training, heads often **specialise**: one follows syntax, another resolves references, another just looks at the previous token.

How many parameters? Each of $W^Q, W^K, W^V, W^O$ has $d_\text{model}^2$ weights, **whatever the number of heads**: more heads means smaller heads, not more parameters.

In [ ]:
for heads_ in [1, 2, 8]:
    m = nn.MultiheadAttention(512, num_heads=heads_, bias=False)
    print(f"d_model=512, h={heads_}: d_k={512 // heads_:3d}, parameters = {sum(p.numel() for p in m.parameters()):,}"
          f"  (4 x 512^2 = {4 * 512 ** 2:,})")

## 6. Positional encoding

Look again at $z_i = \sum_j \alpha_{ij} v_j$: the sum does not depend on **where** token $j$ is. *The dog bit the man* and *the man bit the dog* contain the same tokens.

**Predict first:** will self-attention give *dog* the same output in both sentences?

In [ ]:
torch.manual_seed(1)
vocab = ["the", "dog", "bit", "man"]
embedding = {w: torch.randn(d_model) for w in vocab}

def embed(sentence):
    return torch.stack([embedding[w] for w in sentence.split()])

s1, s2 = "the dog bit the man", "the man bit the dog"
Z1, _ = self_attention(embed(s1), W_q, W_k, W_v)
Z2, _ = self_attention(embed(s2), W_q, W_k, W_v)

print("z_dog in sentence 1:", Z1[1])
print("z_dog in sentence 2:", Z2[4])
print("same?", torch.allclose(Z1[1], Z2[4], atol=1e-6))

# Shuffling the input rows only shuffles the output rows (permutation equivariance)
perm = torch.tensor([4, 2, 0, 3, 1])
Z_perm, _ = self_attention(X5[perm], W_q, W_k, W_v)
print("\nself_attention(X[perm]) == self_attention(X)[perm]?", torch.allclose(Z_perm, Z5[perm], atol=1e-6))

RNNs know the order because they read one token at a time. Transformers must be **told**: a position vector is added to each token embedding, $x_i = \text{embedding}(w_i) + \text{PE}(i)$. The original Transformer uses sinusoids with different frequencies:

$$\text{PE}(pos, 2i) = \sin\left(pos / 10000^{2i/d_\text{model}}\right), \qquad \text{PE}(pos, 2i+1) = \cos\left(pos / 10000^{2i/d_\text{model}}\right)$$

In [ ]:
def positional_encoding(n_positions, d_model):
    pos = torch.arange(n_positions).unsqueeze(1)                 # (n, 1)
    i2 = torch.arange(0, d_model, 2)                             # 2i = 0, 2, 4, ...
    angle = pos / 10000 ** (i2 / d_model)                        # (n, d_model/2)
    PE = torch.zeros(n_positions, d_model)
    PE[:, 0::2] = torch.sin(angle)
    PE[:, 1::2] = torch.cos(angle)
    return PE

PE = positional_encoding(50, 16)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].imshow(PE.T, cmap="RdBu", aspect="auto")
axes[0].set_xlabel("position")
axes[0].set_ylabel("dimension")
axes[0].set_title("PE, $d_{model}$ = 16")
for dim in [0, 2, 6, 14]:
    axes[1].plot(PE[:, dim], label=f"dim {dim}")
axes[1].axvline(5, ls="--", color="gray")
axes[1].set_xlabel("position")
axes[1].set_title("even dimensions: waves with different frequencies")
axes[1].legend(fontsize=8)
PE128 = positional_encoding(50, 128)
im = axes[2].imshow(PE128 @ PE128.T, cmap="viridis")
axes[2].set_xlabel("position")
axes[2].set_ylabel("position")
axes[2].set_title("PE(i) · PE(j), $d_{model}$ = 128")
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()
print("PE(5) =", PE[5].numpy())

**Explain:**

- Low dimensions change fast, high dimensions change slowly, like the digits of a clock. Each position gets a **unique** vector.
- In the right plot, the bands are parallel to the diagonal: $\text{PE}(i) \cdot \text{PE}(j)$ depends mostly on the **distance** $|i - j|$, and nearby positions are similar.

Now with the positions added, does *dog* still get the same output in both sentences?

In [ ]:
PE8 = positional_encoding(10, d_model)
Z1p, _ = self_attention(embed(s1) + PE8[:5], W_q, W_k, W_v)
Z2p, _ = self_attention(embed(s2) + PE8[:5], W_q, W_k, W_v)
print("z_dog in sentence 1:", Z1p[1])
print("z_dog in sentence 2:", Z2p[4])
print("same?", torch.allclose(Z1p[1], Z2p[4], atol=1e-6))

## 7. The Transformer block

A block has two sub-layers, each wrapped by a **residual connection** and **layer normalisation**:

- **self-attention**: tokens exchange information;
- **feed-forward network** (FFN): each token is processed on its own, by the same 2-layer MLP, $\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$, with $d_\text{ff} = 4\,d_\text{model}$.

**Layer normalisation** normalises the vector of **each token** to mean 0 and variance 1, then rescales it with learned $\gamma$ and $\beta$:

$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sigma} + \beta$$

In [ ]:
x = X5[0]
mu, sigma = x.mean(), x.std(unbiased=False)
by_hand = (x - mu) / torch.sqrt(sigma ** 2 + 1e-5)      # gamma = 1, beta = 0 at initialisation

ln = nn.LayerNorm(d_model)
print("by hand     :", by_hand)
print("nn.LayerNorm:", ln(x).detach())
print("mean:", by_hand.mean().item(), " std:", by_hand.std(unbiased=False).item())

The block below uses the **pre-norm** arrangement of most current models (normalise *before* each sub-layer); the original paper normalised *after* the sum (post-norm). The residual connection means that each sub-layer only learns a **correction** to its input: $x \mapsto x + \text{Sublayer}(x)$.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, h, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        y = self.norm1(x)
        x = x + self.attention(y, y, y, need_weights=False)[0]   # tokens exchange information
        x = x + self.ffn(self.norm2(x))                           # each token on its own
        return x

d_model_big, h_big, N = 512, 8, 6
block = TransformerBlock(d_model_big, h_big, 4 * d_model_big)
stack = nn.Sequential(*[TransformerBlock(d_model_big, h_big, 4 * d_model_big) for _ in range(N)])

x_in = torch.randn(1, 10, d_model_big)                            # (batch, n, d_model)
print("input:", tuple(x_in.shape), "-> one block:", tuple(block(x_in).shape), f"-> {N} blocks:", tuple(stack(x_in).shape))

count = lambda m: sum(p.numel() for p in m.parameters())
total = count(block)
print(f"\nparameters of one block: {total:,}")
for name, part in [("attention", block.attention), ("feed-forward", block.ffn), ("layer norms", nn.ModuleList([block.norm1, block.norm2]))]:
    print(f"  {name:>12s}: {count(part):>9,}  ({count(part) / total:.0%})")
print(f"parameters of {N} blocks: {count(stack):,}")

**Explain:**

- Input and output have the same shape, $n \times d_\text{model}$: that is why blocks can be **stacked**.
- The FFN holds about **two thirds** of the parameters: attention has $4\,d_\text{model}^2$ weights, the FFN $2 \cdot d_\text{model} \cdot 4\,d_\text{model} = 8\,d_\text{model}^2$.
- Attention moves information **between** tokens; the FFN transforms the information **within** each token.

## 8. The cost: quadratic in the sequence length

Every token is compared with every token: one head of one layer computes $n^2$ scores.

**Predict first:** if the sequence length doubles, how much longer does the attention take?

In [ ]:
lengths = [256, 512, 1024, 2048, 4096]
times = []
d_head = 64
for n_tok in lengths:
    Qn, Kn, Vn = (torch.randn(n_tok, d_head) for _ in range(3))
    start = time.perf_counter()
    for _ in range(3):
        F.softmax(Qn @ Kn.T / math.sqrt(d_head), dim=-1) @ Vn
    times.append((time.perf_counter() - start) / 3)
    print(f"n = {n_tok:5d}   scores = {n_tok ** 2:>12,}   matrix A: {n_tok ** 2 * 4 / 2 ** 20:7.1f} MB   time: {times[-1] * 1000:7.1f} ms")

plt.figure(figsize=(5, 3.5))
plt.loglog(lengths, times, "o-", label="measured")
plt.loglog(lengths, [times[-1] * (l / lengths[-1]) ** 2 for l in lengths], "--", label="∝ n²")
plt.xlabel("sequence length n")
plt.ylabel("time (s)")
plt.legend()
plt.show()

for n_tok in [512, 4096, 128000]:
    print(f"n = {n_tok:>7,}: n² = {n_tok ** 2:.1e} scores per head and per layer")

**Explain:** doubling $n$ multiplies the number of scores and the memory of $A$ by exactly 4. The measured time usually grows a bit less on a CPU (small matrices do not keep all the cores busy), and gets closer to 4× as $n$ grows. With many heads, many layers and long texts, this is what limits the **context window** of language models.

## 9. Exercises (from the slides)

Solve them on paper first, then check your answers with the cells below.

### Exercise 1: self-attention by hand

With $d_k = 4$, the query of token 3 and the keys and values of a 3-token sequence:

$$q_3 = (1, 0, 1, 0), \quad k_1 = (2, 0, 2, 0), \quad k_2 = (0, 1, 0, 1), \quad k_3 = (1, 1, 1, 1)$$
$$v_1 = (1, 0), \quad v_2 = (0, 1), \quad v_3 = (1, 1)$$

1. Compute the scaled scores $q_3 \cdot k_j / \sqrt{d_k}$.
2. Compute the attention weights $\alpha_{31}, \alpha_{32}, \alpha_{33}$.
3. Compute the output $z_3$.
4. Which token does token 3 attend to most? Why?

In [ ]:
q3 = np.array([1, 0, 1, 0])
K_ex = np.array([[2, 0, 2, 0], [0, 1, 0, 1], [1, 1, 1, 1]])
V_ex = np.array([[1, 0], [0, 1], [1, 1]])

s_ex = K_ex @ q3 / np.sqrt(4)
alpha_ex = softmax_rows(s_ex[None, :])[0]
print("dot products:", K_ex @ q3, "  scaled scores:", s_ex)
print("exp:", np.exp(s_ex), " sum:", np.exp(s_ex).sum())
print("alpha:", alpha_ex)
print("z_3 =", alpha_ex @ V_ex)

### Exercise 2: causal mask

The scaled scores of a 3-token sequence (rows: queries, columns: keys):

$$S = \begin{bmatrix} 2 & 1 & 0 \\ 1 & 3 & 1 \\ 0 & 2 & 2 \end{bmatrix}$$

1. Apply the causal mask to $S$.
2. Compute the attention matrix $A$ (softmax of each row).
3. Which tokens can token 1 see? Why is its row trivial?
4. What would go wrong in training without the mask? (The `transformer_translation` notebook tries it.)

In [ ]:
S_ex = np.array([[2.0, 1, 0], [1, 3, 1], [0, 2, 2]])
M_ex = np.where(np.tril(np.ones((3, 3))) == 1, 0, -np.inf)
print("S + M =\n", S_ex + M_ex)
print("A =\n", softmax_rows(S_ex + M_ex))

### Exercise 3: sizes and cost

A multi-head attention layer with $d_\text{model} = 512$ and $h = 8$ heads:

1. What is $d_k$ for each head?
2. How many weights are in $W^Q, W^K, W^V$ (all heads together) and $W^O$? Ignore biases.
3. How many attention weights $\alpha_{ij}$ does one head compute for $n = 1000$ tokens? And for all heads?
4. By how much does this grow for $n = 2000$?

In [ ]:
mha_ex = nn.MultiheadAttention(512, num_heads=8, bias=False, batch_first=True)
print("1) d_k =", 512 // 8)
print(f"2) parameters of nn.MultiheadAttention(512, 8, bias=False): {sum(p.numel() for p in mha_ex.parameters()):,}")

x_ex = torch.randn(1, 1000, 512)
with torch.no_grad():
    _, A_ex = mha_ex(x_ex, x_ex, x_ex, average_attn_weights=False)
print(f"3) attention weights for n=1000: {tuple(A_ex.shape)} -> {A_ex[0, 0].numel():,} per head, {A_ex.numel():,} in total")
print(f"4) n=2000: {2000 ** 2:,} per head, {2000 ** 2 / 1000 ** 2:.0f} times more")

## 10. Back to *it*: self-attention in a pre-trained model

The opening exercise of the slides:

1. *The animal didn't cross the street because **it** was too **tired**.*
2. *The animal didn't cross the street because **it** was too **wide**.*

We load **BERT** (`bert-base-uncased`), an encoder-only Transformer with $N = 12$ blocks, $h = 12$ heads, $d_\text{model} = 768$, pre-trained by predicting hidden words. We ask it for all its attention matrices and all the token vectors after each block. (The first run downloads about 440 MB.)

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers
    import transformers

from transformers import AutoModel, AutoTokenizer

transformers.logging.set_verbosity_error()      # hide the loading report
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager").eval()

sentences = {"tired": "The animal didn't cross the street because it was too tired.",
             "wide": "The animal didn't cross the street because it was too wide."}
results, bert_tokens = {}, {}
for key, sentence in sentences.items():
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = bert(**inputs, output_attentions=True, output_hidden_states=True)
    results[key] = out
    bert_tokens[key] = tokenizer.convert_ids_to_tokens(inputs.input_ids[0])

tokens_tired = bert_tokens["tired"]
i_it, i_animal, i_street = tokens_tired.index("it"), tokens_tired.index("animal"), tokens_tired.index("street")
print(tokens_tired)
print("attentions:", len(out.attentions), "layers of shape", tuple(out.attentions[0].shape), "(batch, heads, n, n)")

### Contextual embeddings

The token *it* is at the same position in both sentences, so it **enters** the network with exactly the same vector. Does it leave with the same vector?

In [ ]:
similarity = [F.cosine_similarity(results["tired"].hidden_states[L][0, i_it],
                                  results["wide"].hidden_states[L][0, i_it], dim=0).item()
              for L in range(len(out.hidden_states))]

plt.figure(figsize=(6, 3.2))
plt.plot(range(len(similarity)), similarity, "o-")
plt.xlabel("after block (0 = input embedding)")
plt.ylabel("cosine similarity")
plt.title("vector of 'it' in the 'tired' vs the 'wide' sentence")
plt.grid(alpha=0.3)
plt.show()

### Does some head resolve *it*?

BERT has $12 \times 12 = 144$ heads. For each one, we measure how much more *it* attends to *animal* than to *street* in the *tired* sentence, and the opposite in the *wide* sentence, and pick the head where this is largest.

In [ ]:
best = None
for L in range(12):
    A_tired = results["tired"].attentions[L][0, :, i_it]     # (heads, n): row of 'it' in every head
    A_wide = results["wide"].attentions[L][0, :, i_it]
    for hd in range(12):
        gap = (A_tired[hd, i_animal] - A_tired[hd, i_street]) + (A_wide[hd, i_street] - A_wide[hd, i_animal])
        if best is None or gap > best[0]:
            best = (gap.item(), L, hd)

_, L_best, h_best = best
rows = np.stack([results[k].attentions[L_best][0, h_best, i_it].numpy() for k in ["tired", "wide"]])
plt.figure(figsize=(12, 2.2))
both = [w if w != "tired" else "tired/wide" for w in tokens_tired]
heatmap(rows, both, ["'it' (tired)", "'it' (wide)"], f"attention of 'it' in layer {L_best + 1}, head {h_best + 1}",
        ax=plt.gca(), vmax=rows.max())
plt.show()
for k in ["tired", "wide"]:
    r = results[k].attentions[L_best][0, h_best, i_it]
    print(f"{k:>5s}: it -> animal {r[i_animal]:.2f}   it -> street {r[i_street]:.2f}")

**Explain:**

- In the *tired* sentence this head sends the attention of *it* to *animal*; in the *wide* sentence, to *street*. Nobody told BERT what *it* refers to: it learned this by predicting hidden words in a lot of text.
- Be careful: we **searched** 144 heads for the one that does what we wanted. Most heads do something else (look at the previous token, at `[SEP]`, ...). Try other layers and heads with the cell below.
- BERT is **bidirectional**: *it* can see *tired* and *wide*, which come later. In a decoder-only model (GPT), *it* cannot see them. Where in the network could the ambiguity still be resolved?

In [ ]:
layer, head = L_best, h_best       # the head found above; try others (0..11), e.g. layer, head = 2, 0
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, k in zip(axes, ["tired", "wide"]):
    heatmap(results[k].attentions[layer][0, head].numpy(), bert_tokens[k], bert_tokens[k],
            f"{k}: layer {layer + 1}, head {head + 1}", ax=ax, fmt="{:.1f}")
plt.tight_layout()
plt.show()

## Summary

- Self-attention: every token builds its new vector from all the tokens of its sequence, $\text{softmax}(QK^\top/\sqrt{d_k})\,V$, all in parallel.
- Dividing by $\sqrt{d_k}$ keeps the softmax away from saturation.
- The causal mask hides future tokens, so generation can be trained in one pass.
- Multi-head attention follows several relations at once, with the same number of parameters.
- Self-attention ignores order; positional encodings add it.
- A Transformer block = attention + feed-forward, each with a residual connection and layer norm, stacked $N$ times.
- The cost grows with $n^2$.

## Your turn

1. Section 1: change $W^Q$ so that *house* attends mostly to *blue*. Can you do it while keeping $W^K = W^V = I$?
2. Section 3: repeat the table with keys and queries of variance 0.1 instead of 1. Is scaling still needed?
3. Section 4: build a mask where each token can see only itself and the **previous two** tokens (local attention). How many scores are really needed for $n$ tokens?
4. Section 6: replace the sinusoidal encoding by a **learned** one (`nn.Embedding(max_len, d_model)`). What happens with a sentence longer than `max_len`?
5. Section 7: change the block to post-norm (`x = norm(x + sublayer(x))`) and check that the shapes and the parameter count do not change.
6. Section 10: find the head where *it* attends most to the **previous** token, and the one that attends most to `[SEP]`.

In [ ]:
# Your turn